In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.sql('select * from vishnudatabrickslatest.silver.customers')

In [0]:
df.display()

In [0]:
dbutils.widgets.text('load_flag','')

In [0]:
load_flag = int(dbutils.widgets.get('load_flag'))

In [0]:
%sql
create schema if not exists vishnudatabrickslatest.gold

In [0]:
if load_flag == 0:
    df_old = spark.sql('''select DimCustomerKey, customer_id, create_date, update_date 
                       from vishnudatabrickslatest.gold.DimCustomers ''')
else:
    df_old = spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date 
                       from vishnudatabrickslatest.silver.customers where 1=0 ''')

In [0]:
df_old.display()

In [0]:
df_old = df_old.withColumnRenamed("customer_id",'old_customer_id')\
                .withColumnRenamed("DimCustomerKey",'old_DimCustomerKey')\
                    .withColumnRenamed("create_date",'old_create_date')\
                    .withColumnRenamed("update_date",'old_update_date')

In [0]:
df_join = df.join(df_old, df['customer_id'] == df_old['old_customer_id'], how='left')

In [0]:
df_join.display()

##Diving new vs old records

In [0]:
df_new = df_join.filter(df_join['old_DimCustomerKey'].isNull())
df_old = df_join.filter(df_join['old_DimCustomerKey'].isNotNull())

In [0]:
df_new.display()

In [0]:
df_old.display()

##Preparing df_old

In [0]:
#Drop unnecessary columns
df_old = df_old.drop('old_customer_id','old_update_date')

#renaming DimCustomer key old to new

df_old = df_old.withColumnRenamed("old_DimCustomerKey",'DimCustomerKey')

#Renaming old_createdate to create date
df_old = df_old.withColumnRenamed("old_create_date",'create_date')
df_old = df_old.withColumn('create_date',to_timestamp(col('create_date')))

#creating new update date column

df_old = df_old.withColumn('update_date',current_timestamp())
df_old.display()


#Preparing df_new

In [0]:
df_new.display()

In [0]:
#Drop unnecessary columns
df_new = df_new.drop('old_DimCustomerKey','old_customer_id','old_create_date','old_update_date')
df_new = df_new.withColumn('create_date',current_timestamp())
df_new = df_new.withColumn('update_date',current_timestamp())

In [0]:
df_new.display()

##Creating max surr key

In [0]:
df_new = df_new.withColumn('DimCustomerKey',monotonically_increasing_id() + 1)

In [0]:
%sql
select max(DimCustomerKey) as max_surr_key from vishnudatabrickslatest.gold.DimCustomers

In [0]:
if load_flag == 1:
    print("Vishnu")
else:
    print('Reddy')

In [0]:
if load_flag == 1:
    max_surr_key = 0
else:
    max_surr_key = spark.sql("""
        SELECT COALESCE(MAX(DimCustomerKey),0) AS max_surr_key
        FROM vishnudatabrickslatest.gold.DimCustomers
    """).collect()[0]["max_surr_key"]

In [0]:
print(max_surr_key)

In [0]:
df_new = df_new.withColumn('DimCustomerKey', lit(max_surr_key)+ col('DimCustomerKey'))

In [0]:
print(max_surr_key)

In [0]:
df_new.display()

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.display()

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("vishnudatabrickslatest.gold.DimCustomers"):
    deltaTable = DeltaTable.forPath(spark,'abfss://gold@datalakrelatestreddy.dfs.core.windows.net/dimcustomers')
    deltaTable.alias('trg').merge(df_final.alias('src'),'trg.DimCustomerKey = src.DimCustomerKey').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format('delta').mode('overwrite').option('path','abfss://gold@datalakrelatestreddy.dfs.core.windows.net/dimcustomers').saveAsTable('vishnudatabrickslatest.gold.DimCustomers')

In [0]:
%sql
select * from vishnudatabrickslatest.gold.dimcustomers